# Aspire : des tests d'intégration modernes — Testcontainers, TUnit, rollback transactionnel

Ce notebook couvre les axes **A5 + A6 + A7** du **Grain 2** de la digestion #11516
(*The Unexpected AI Stack: C# + .NET*, Part 4) :

| Axe | Apport | Où le voir |
|---|---|---|
| **A5** | `Testcontainers.PostgreSql` : un Postgres 18 **jetable**, port hôte aléatoire, wait strategy | §2 |
| **A6** | **TUnit** + **Microsoft Testing Platform** (MTP) : paradigme de test absent du dépôt (xUnit/vitest partout) | §1 |
| **A7** | Isolation par **rollback transactionnel** : chaque test repart d'une base intacte, en parallèle | §3 |

Le livrable à côté de ce notebook : le projet [`IntegrationTests/`](IntegrationTests/) —
exécutable tel quel (`dotnet test`), Docker requis.

## Contexte : xUnit partout, et une question d'état

La sonde de l'issue #11516 sur tout le dépôt ne trouve **aucun** usage réel de TUnit ni de
Testcontainers : les tests du repo sont en xUnit (.NET) et vitest (TS). Ce n'est pas un
problème en soi — mais la Part 4 de la série source démontre un **autre contrat de test**,
conçu pour des agents et des humains :

1. **pas de base installée à la main** : le conteneur Postgres se lance avec le test, sur un
   port **aléatoire** — plusieurs sessions coexistent sans collision ;
2. **pas d'état résiduel** : chaque test s'exécute dans **sa** transaction, ouverte avant et
   *rollbackée* après — la base est intangible, les tests peuvent tourner en parallèle ;
3. **un runner moderne** : MTP remplace l'ancien pipeline VSTest ; le projet de test est un
   exécutable, filtrable par **arbre** (`--treenode-filter`).

Chaque brique est montrée **réellement exécutée** ci-dessous : le conteneur démarre sous vos
yeux dans les sorties de `dotnet test`.

In [1]:
using System.IO;
// Cellule d'amorcage : chemins partages + helper d'execution.
public static class TestShell {
    public static readonly string RepoRoot = FindRepoRoot(Directory.GetCurrentDirectory());
    public static readonly string ProjectDir = Path.Combine(
        RepoRoot, "MyIA.AI.Notebooks", "GenAI", "Aspire", "IntegrationTests");

    public static string FindRepoRoot(string start) {
        var dir = new DirectoryInfo(start);
        while (dir != null && !Directory.Exists(Path.Combine(dir.FullName, "docker-configurations")))
            dir = dir.Parent;
        return dir?.FullName ?? throw new DirectoryNotFoundException("racine du depot introuvable");
    }

    // Execute une commande et capture stdout + stderr concatenees.
    public static string Run(string workDir, string file, string args, int timeoutMs = 600_000) {
        var psi = new System.Diagnostics.ProcessStartInfo {
            FileName = file, Arguments = args,
            WorkingDirectory = workDir,
            RedirectStandardOutput = true, RedirectStandardError = true,
            UseShellExecute = false, CreateNoWindow = true
        };
        using var p = System.Diagnostics.Process.Start(psi)!;
        var stdout = p.StandardOutput.ReadToEnd();
        var stderr = p.StandardError.ReadToEnd();
        if (!p.WaitForExit(timeoutMs)) { p.Kill(); return "[TIMEOUT] " + stdout + stderr; }
        return stdout + stderr;
    }

    public static string Dotnet(string args) => Run(ProjectDir, "dotnet.exe", args);
    public static string Docker(string args) => Run(RepoRoot, "docker.exe", args, 120_000);

    // Projection d'affichage : le resume MTP embarque le chemin absolu de la
    // DLL — on montre les lignes de resume sans la ligne qui le porte.
    public static string Clean(string s) =>
        string.Join(Environment.NewLine,
            s.Split(Environment.NewLine).Where(l => !l.Contains(RepoRoot)));


    // Affiche un fichier source du projet, titre en tete.
    public static object ShowFile(string relativePath) {
        var full = Path.Combine(ProjectDir, relativePath);
        var content = File.ReadAllText(full);
        return display($"--- {relativePath} ---\n{content}");
    }
}
display($"Projet (relatif a la racine du depot) : {Path.GetRelativePath(TestShell.RepoRoot, TestShell.ProjectDir)}");
display(File.Exists(Path.Combine(TestShell.ProjectDir, "IntegrationTests.csproj"))
    ? "Projet IntegrationTests present." : "PROJET ABSENT !");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Projet (relatif a la racine du depot) : MyIA.AI.Notebooks\GenAI\Aspire\IntegrationTests

Projet IntegrationTests present.

## 1. A6 — TUnit + Microsoft Testing Platform : le harnais

Sur le SDK .NET 10, l'ancien pont VSTest de `dotnet test` n'existe plus : MTP devient le
runner natif. Deux réglages suffisent — une propriété csproj, et le `global.json` qui
déclare le runner pour l'expérience `dotnet test` nouvelle génération.

In [2]:
TestShell.ShowFile("IntegrationTests.csproj");
TestShell.ShowFile("global.json");
TestShell.ShowFile("SmokeTest.cs");

--- IntegrationTests.csproj ---
<Project Sdk="Microsoft.NET.Sdk">

  <PropertyGroup>
    <OutputType>Exe</OutputType>
    <TargetFramework>net10.0</TargetFramework>
    <ImplicitUsings>enable</ImplicitUsings>
    <Nullable>enable</Nullable>
    <IsPackable>false</IsPackable>
    <!-- Microsoft Testing Platform : `dotnet test` delegue au runner TUnit -->
    <TestingPlatformDotnetTestSupport>true</TestingPlatformDotnetTestSupport>
  </PropertyGroup>

  <ItemGroup>
    <PackageReference Include="EFCore.NamingConventions" Version="10.0.1" />
    <PackageReference Include="Npgsql.EntityFrameworkCore.PostgreSQL" Version="10.0.3" />
    <PackageReference Include="Testcontainers.PostgreSql" Version="4.14.0" />
    <PackageReference Include="TUnit" Version="1.65.0" />
  </ItemGroup>

</Project>


--- global.json ---
{
  "test": {
    "runner": "Microsoft.Testing.Platform"
  }
}


--- SmokeTest.cs ---
using TUnit.Core;

namespace IntegrationTests;

/// <summary>
/// Fumigenne : prouve que le harnais TUnit + Microsoft Testing Platform est
/// cable (sans base) — premier reflexe quand on echafaude un projet de tests.
/// </summary>
public class SmokeTest
{
    [Test]
    public async Task SmokeTest_Harness_IsWired()
    {
        // Valeur non constante (le linter TUnit refuse les assertions
        // constantes) : si cette ligne passe, le runner a bien execute du
        // code utilisateur.
        await Assert.That(DateTime.UtcNow.Year).IsGreaterThanOrEqualTo(2026);
    }
}


In [3]:
// La fumigenne, via le runner NATIF (executable MTP) : dotnet run -- <args>.
// Le pont `dotnet test` accepte les executions completes, mais refuse les filtres
// sur .NET 10 — le runner natif les accepte tous.
var smoke = TestShell.Dotnet("run --no-build -- --treenode-filter \"/IntegrationTests/IntegrationTests/SmokeTest/*\"");
display(TestShell.Clean(smoke[smoke.LastIndexOf("Résumé", StringComparison.Ordinal)..]));

  total: 1
  échec: 0
  opération réussie: 1
  ignoré: 0
  durée: 330ms


### Interprétation

- **`total: 1`** : le filtre d'arbre `/Assembly/Namespace/Classe/*` a sélectionné exactement
  la classe `SmokeTest` — sans toucher au conteneur (ce test ne touche pas la base).
- Le runner affiche `opération réussie: 1` : `Assert.That(...)` (fluent, asynchrone) est le
  style d'assertion TUnit, distinct du `Assert.True(...)` xUnit.
- `dotnet run --` est la forme **native** : le projet de test est un exécutable MTP ; tous
  ses arguments (`--treenode-filter`, `--list-tests`, `--diagnostic`) passent après `--`.

In [4]:
// EXERCICE 1 : executer UNE methode precise.
// Objectif : lancer uniquement SmokeTest_Harness_IsWired via --treenode-filter,
// et verifier que le resume affiche total: 1 (et non 5).
// Indice : le dernier segment accepte un joker final — SmokeTest_Harness_IsWired*
// Etape 1 : construire le filtre "/IntegrationTests/IntegrationTests/SmokeTest/SmokeTest_Harness_IsWired*"
// Etape 2 : le passer a TestShell.Dotnet("run --no-build -- --treenode-filter \"...\"")
// Etape 3 : afficher le resume (dernieres lignes) et verifier total: 1
Console.WriteLine("Exercice 1 a completer : filtrer sur une methode unique et verifier total: 1");

Exercice 1 a completer : filtrer sur une methode unique et verifier total: 1


## 2. A5 — Testcontainers : un Postgres 18 jetable, port aléatoire

La fixture ne demande **rien** d'installé : elle tire l'image `postgres:18`, démarre le
conteneur sur un **port hôte aléatoire** (jamais 5432 en dur — plusieurs sessions de test
coexistent), attend la double condition de santé (message de log **et** port TCP interne),
crée le schéma, puis s'auto-détruit (`WithAutoRemove`).

In [5]:
TestShell.ShowFile("PgDatabaseFixture.cs");

--- PgDatabaseFixture.cs ---
using DotNet.Testcontainers.Builders;
using IntegrationTests.Data;
using Microsoft.EntityFrameworkCore;
using Testcontainers.PostgreSql;
using TUnit.Core.Interfaces;

namespace IntegrationTests;

/// <summary>
/// Fixture TUnit : demarre UN conteneur Postgres 18 par session de tests,
/// sur un port hote aleatoire (jamais 5432 en dur), puis cree le schema.
/// Un seul conteneur pour toute la session : les tests s'executent en
/// parallele dessus, isoles par rollback transactionnel (cf.
/// <see cref="PgTransactionalTestBase"/>).
/// </summary>
public class PgDatabaseFixture : IAsyncInitializer, IAsyncDisposable
{
    private PostgreSqlContainer? _container;
    private DbContextOptions<TranscriptionDbContext>? _options;

    private DbContextOptions<TranscriptionDbContext> EnsureOptions()
    {
        if (_container is null)
        {
            throw new InvalidOperationException("Le conteneur Postgres n'est pas initialise.");
        }

        return 

In [6]:
// L'execution COMPLETE : dotnet test lance le conteneur, joue les 5 tests, tout disparait.
var full = TestShell.Dotnet("test");
display(TestShell.Clean(full[full.LastIndexOf("Exécuter des tests", StringComparison.Ordinal)..]));


  Artéfacts produits dans les dossiers en cours de traitement :

Résumé de série de tests : Réussite!
  total : 5
  échec : 0
  réussie : 5
  ignoré : 0
  durée : 10s 179ms


In [7]:
// Preuve de l'ephemerite : apres le run, plus AUCUN conteneur postgres residuel.
var residue = TestShell.Docker("ps -a --filter ancestor=postgres:18 --format \"{{.ID}} {{.Image}} {{.Status}}\"");
display(string.IsNullOrWhiteSpace(residue)
    ? "(vide) : aucun conteneur postgres:18 residuel — WithAutoRemove a tout nettoye."
    : residue);

(vide) : aucun conteneur postgres:18 residuel — WithAutoRemove a tout nettoye.

### Interprétation

- **`réussie: 5`** en ~10 s à chaud (comptez ~30 s au premier run, tirage de l'image
  compris) — rien à installer, rien à nettoyer après (sortie `(vide)` ci-dessus).
- **`WithPortBinding(5432, true)`** : le `true` demande un port hôte **aléatoire** ; la
  chaîne de connexion lue par `GetConnectionString()` pointe dessus. C'est la fin des
  collisions de port entre worktrees — le même principe que `aspire run --isolated`
  (notebook 01), au niveau du test.
- **Un seul conteneur par session** : `[ClassDataSource<PgDatabaseFixture>(Shared =
  SharedType.PerTestSession)]` — le démarrage est payé une fois, les tests tournent
  dessus en parallèle.
- La **wait strategy** est double (`UntilMessageIsLogged` + `UntilInternalTcpPortIsAvailable`) :
  le port peut écouter avant que Postgres ait fini son init — le message de log est la
  vraie garantie.

In [8]:
// EXERCICE 2 : mesurer le cout du conteneur.
// Objectif : chronometrer le smoke test SANS base (filtre SmokeTest) vs les tests
// AVEC base (filtre TranscriptionJobTests), et calculer le surcout conteneur.
// Indice : System.Diagnostics.Stopwatch autour de deux TestShell.Dotnet(...)
// Etape 1 : mesurer le run filtre SmokeTest (pas de conteneur)
// Etape 2 : mesurer le run filtre TranscriptionJobTests (conteneur + 4 tests)
// Etape 3 : afficher les deux durees et leur difference
Console.WriteLine("Exercice 2 a completer : chronometrer sans-base vs avec-base");

Exercice 2 a completer : chronometrer sans-base vs avec-base


## 3. A7 — l'isolation par rollback : chaque test repart de zéro

Le couple `PgTransactionalTestBase` + tests : **avant** chaque test on ouvre une
transaction, **après** on la rollback. Un test qui écrit ne pollue jamais le suivant —
l'invariant « la table est vide au début d'un test » tient même en parallèle, parce que
les lignes non commises sont invisibles aux autres transactions.

In [9]:
TestShell.ShowFile("PgTransactionalTestBase.cs");
TestShell.ShowFile("TranscriptionJobTests.cs");

--- PgTransactionalTestBase.cs ---
using IntegrationTests.Data;
using Microsoft.EntityFrameworkCore.Storage;
using TUnit.Core;

namespace IntegrationTests;

/// <summary>
/// Base transactionnelle : chaque test s'execute dans SA transaction, ouverte
/// en [Before(Test)] et ROLLBACKEE en [After(Test)]. Aucun test ne laisse de
/// ligne derriere lui — l'etat de depart (base vide) est un invariant, et les
/// tests peuvent tourner en parallele sur le meme conteneur.
/// </summary>
public abstract class PgTransactionalTestBase(PgDatabaseFixture pg)
{
    private IDbContextTransaction? _transaction;

    // TUnit instancie la classe pour CHAQUE test : le champ est frais a chaque fois
    protected TranscriptionDbContext Context = pg.CreateContext();

    [Before(Test)]
    public async Task BeginTransaction()
    {
        _transaction = await Context.Database.BeginTransactionAsync();
    }

    [After(Test)]
    public async Task RollbackTransaction()
    {
        if (_transaction is not

--- TranscriptionJobTests.cs ---
using IntegrationTests.Data;
using Microsoft.EntityFrameworkCore;
using TUnit.Core;

namespace IntegrationTests;

/// <summary>
/// Tests d'integration EF Core + Postgres reel (conteneur Testcontainers),
/// isoles par rollback transactionnel. Convention de nommage trois parties :
/// Entite_Etat_Testee_Comportement_Attendu.
/// </summary>
[ClassDataSource<PgDatabaseFixture>(Shared = SharedType.PerTestSession)]
public class TranscriptionJobTests(PgDatabaseFixture pg) : PgTransactionalTestBase(pg)
{
    [Test]
    public async Task TranscriptionJob_WriteThenRead_RoundTrips()
    {
        Context.Jobs.Add(new TranscriptionJob
        {
            FileName = "echantillon-test-fr.wav",
            Model = "faster-whisper-large-v3-turbo",
            DurationSeconds = 12.5,
            Status = "Done",
        });

        await Context.SaveChangesAsync();

        Context.ChangeTracker.Clear(); // relecture forcee depuis la base

        var job = await C

In [10]:
// La preuve de coexistence : on execute ENSEMBLE les quatre tests de
// TranscriptionJobTests — dont celui qui ECRIT une ligne (WriteThenRead_RoundTrips)
// et celui qui AFFIRME la table vide (AfterEachRollback_TableIsStillEmpty).
var pair = TestShell.Dotnet("run --no-build -- --treenode-filter \"/IntegrationTests/IntegrationTests/TranscriptionJobTests/TranscriptionJob_*\"");
display(TestShell.Clean(pair[pair.LastIndexOf("Résumé", StringComparison.Ordinal)..]));

  total: 4
  échec: 0
  opération réussie: 4
  ignoré: 0
  durée: 10s 747ms


### Interprétation

- **`total: 4`** : les quatre tests passent **ensemble**, dont `WriteThenRead_RoundTrips`
  (qui insère une ligne) et `AfterEachRollback_TableIsStillEmpty` (qui compte zéro ligne).
  C'est la démonstration A7 : le rollback de l'un rend son écriture invisible à l'autre —
  l'ordre d'exécution n'existe plus comme source de vérité.
- `Context.ChangeTracker.Clear()` avant relecture : force EF Core à recharger **depuis la
  base** (dans la transaction du test), pas depuis son cache mémoire.
- **`DuplicateFileName_RejectedByUniqueIndex`** : la contrainte UNIQUE posée dans
  `TranscriptionJobConfiguration` est vérifiée **par le vrai serveur Postgres** — c'est un
  test d'intégration, pas un mock.
- Convention de nommage trois parties : `Entite_Etat_Teste_Comportement_Attendu`.

In [11]:
// EXERCICE 3 : l'invisibilite inter-connections.
// Objectif : montrer qu'un deuxieme contexte ne VOIT PAS les lignes non commises
// du premier (lecture sale impossible en READ COMMITTED) — au-dela du rollback.
// Indice : ajouter a TranscriptionJobTests une methode TwoContexts_Uncommitted_Invisible
// qui cree un second contexte via pg.CreateContext(), ecrit+SaveChanges dans le premier,
// puis compte dans le second AVANT tout commit.
// Etape 1 : ecrire le test dans TranscriptionJobTests.cs
// Etape 2 : le faire passer via --treenode-filter (cf. exercice 1)
// Etape 3 : expliquer pourquoi le rollback reste necessaire malgre cette invisibilite
Console.WriteLine("Exercice 3 a completer : demonstrer l'invisibilite inter-connections");

Exercice 3 a completer : demonstrer l'invisibilite inter-connections


## Conclusion

Le harnais complet tient en quatre fichiers et un contrat :

| Brique | Fichier | Garantie |
|---|---|---|
| Runner TUnit + MTP | `IntegrationTests.csproj` + `global.json` | exécutable filtrable, `dotnet test` natif .NET 10 |
| Fumigenne | `SmokeTest.cs` | le harnais est câblé avant d'échafauder |
| Conteneur jetable | `PgDatabaseFixture.cs` | Postgres 18 réel, port aléatoire, auto-purge |
| Isolation | `PgTransactionalTestBase.cs` | rollback par test, parallélisme sans ordre |

Ce que ce projet **n'est pas** : un remplacement des xUnit du dépôt — c'est la démonstration
d'un paradigme complémentaire (Part 4 de la série source), self-contained à côté des
notebooks de la série Aspire. La suite logique de la digestion #11516 : le grain 1
(observabilité Serilog + OTel, notebook 03) et le grain 3 (agent streaming, notebook 04).

**Prérequis** pour rejouer : SDK .NET 10, Docker démarré (image `postgres:18` tirée au
premier run), puis `dotnet test` dans `IntegrationTests/`.